# 🧠 AeroSync: Geo-Cadastral RAG & LLM Land Intelligence Suite
**Problem Statement ID: 26012 | DoLR, Ministry of Rural Development**
**Architecture: Cadastral Knowledge Store + Spatial GeoJSON Retrieval + Multi-Backend LLM Assistant**

## Step 0: Auto-Install Dependencies

In [ ]:
import sys, subprocess, importlib
pkgs = ['pandas', 'numpy', 'scikit-learn']
for p in pkgs:
    try:
        importlib.import_module(p.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])
print('[OK] AeroSync RAG & LLM Engine Dependencies Ready.')

## Step 1: Setup & Imports

In [ ]:
import os, sys, json, math, subprocess, importlib
import numpy as np, pandas as pd

# ── AeroSync workspace resolver (Colab / Kaggle / Local Auto-Detection) ───────
_candidates = [
    os.getcwd(),
    r"C:\AeroSync",
    "/content/AeroSync",
    "/content",
    "/kaggle/working/AeroSync",
    "/kaggle/working",
    os.path.abspath(".."),
]

workspace_dir = next(
    (p for p in _candidates if p and os.path.exists(os.path.join(p, "models"))),
    None,
)

# Auto-clone or pull latest repository if running in Google Colab / Kaggle
if workspace_dir is None:
    print("[INFO] 'models' module not found locally. Auto-cloning AeroSync repository...")
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/thatvivekhingu/AeroSync.git"],
            check=True,
        )
        for _p in ["AeroSync", "/content/AeroSync", "/kaggle/working/AeroSync"]:
            if os.path.exists(os.path.join(_p, "models")):
                workspace_dir = os.path.abspath(_p)
                break
    except Exception as _e:
        print(f"[WARNING] Could not auto-clone repository: {_e}")
else:
    if os.path.exists(os.path.join(workspace_dir, ".git")) and ("/content" in workspace_dir or "/kaggle" in workspace_dir):
        try:
            subprocess.run(["git", "-C", workspace_dir, "pull"], check=False)
        except Exception:
            pass

if workspace_dir is None:
    workspace_dir = os.getcwd()

if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

if "models" in sys.modules:
    importlib.reload(sys.modules["models"])

from models import (
    CadastralKnowledgeBase,
    SpatialGeoJSONRetriever,
    AeroSyncCadastralLLM,
    audit_regulatory_compliance,
    generate_property_card,
    CLASS_NAMES,
)

print(f"[OK] AeroSync RAG Engine loaded | Workspace: {workspace_dir}")

## Step 2: Initialize SVAMITVA Legal Knowledge Base

In [ ]:
kb = CadastralKnowledgeBase()
print(f"[OK] Indexed {len(kb.documents)} Legal & Technical SVAMITVA Guidelines.")

# Demonstration Retrieval
query = "What is ULPIN Bhu-Aadhaar and how is it generated?"
retrieved = kb.retrieve(query, top_k=2)
print(f"\n=== Top Retrieved Documents for Query: '{query}' ===")
for r in retrieved:
    print(f"\n📌 [{r['title']}] (Score: {r['relevance_score']:.3f})")
    print(f"   {r['content'][:180]}...")

## Step 3: Ingest Drone AI GeoJSON Survey Output

In [ ]:
# Simulated or Real Survey GeoJSON
demo_survey_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [[[82.973, 25.317], [82.974, 25.317], [82.974, 25.318], [82.973, 25.318], [82.973, 25.317]]]}
            ,
            "properties": {
                "ulpin": "ULPIN-26012-0042",
                "class_id": 1,
                "class_name": "Building",
                "area_sqm": 145.5,
                "perimeter_m": 48.2,
                "confidence": 0.94,
                "surveyor_verification_needed": False,
            }
        },
        {
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [[[82.975, 25.319], [82.976, 25.319], [82.976, 25.320], [82.975, 25.320], [82.975, 25.319]]]}
            ,
            "properties": {
                "ulpin": "ULPIN-26012-0043",
                "class_id": 1,
                "class_name": "Building",
                "area_sqm": 92.0,
                "perimeter_m": 38.6,
                "confidence": 0.61,
                "surveyor_verification_needed": True,
            }
        },
        {
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [[[82.9735, 25.3175], [82.9745, 25.3175], [82.9745, 25.3185], [82.9735, 25.3185], [82.9735, 25.3175]]]}
            ,
            "properties": {
                "ulpin": "ULPIN-26012-0099",
                "class_id": 3,
                "class_name": "Water",
                "area_sqm": 450.0,
                "perimeter_m": 90.0,
                "confidence": 0.98,
                "surveyor_verification_needed": False,
            }
        }
    ]
}

spatial_retriever = SpatialGeoJSONRetriever(demo_survey_geojson)
stats = spatial_retriever.get_summary_stats()
print("=== Drone Survey Spatial Summary ===")
for k, v in stats.items():
    print(f"  • {k}: {v}")

## Step 4: Proximity & Regulatory Buffer Violation Audit

In [ ]:
violations = audit_regulatory_compliance(spatial_retriever.parcels, water_buffer_m=15.0, road_setback_m=3.0)
print(f"[AUDIT] Total Regulatory Issues Detected: {len(violations)}")
for idx, v in enumerate(violations, 1):
    print(f"\n🚨 Issue #{idx}: {v['violation_type']} [{v['severity']}]")
    print(f"   Parcel ULPIN: {v['parcel_ulpin']}")
    print(f"   Distance to Water Body / Road: {v['distance_m']}m (Permitted Min: {v['permitted_min_m']}m)")
    print(f"   Remedial Action: {v['remedial_action']}")

## Step 5: Automated SVAMITVA Property Card (Gharoni) Generator

In [ ]:
target_parcel = spatial_retriever.find_by_ulpin("0042")
if target_parcel:
    card = generate_property_card(
        parcel=target_parcel,
        state="Uttar Pradesh",
        district="Varanasi",
        tehsil="Pindra",
        village="Babatpur",
        owner_name="Smt. Sunita Devi & Shri Rameshwar Patel",
    )
    print(json.dumps(card, indent=2))
else:
    print("Parcel not found.")

## Step 6: Interactive AeroSync Cadastral AI Chat Assistant (English & Hindi)

In [ ]:
# Multi-Backend: Works with Gemini API (if key present) or built-in Offline Reasoner
llm_assistant = AeroSyncCadastralLLM(
    knowledge_base=kb,
    spatial_retriever=spatial_retriever,
    provider="auto",
)
print(f"[OK] AeroSync LLM initialized with backend: '{llm_assistant.backend_type.upper()}'\n")

# Sample Questions
sample_queries = [
    "Gaon me kitne residential houses detect hue hain aur total area kitna hai?",
    "What is the protocol if a parcel has low AI confidence and high uncertainty?",
    "Check if any house is violating water body buffer zone near pond.",
    "Generate draft Property Card for ULPIN-26012-0042.",
]

for q in sample_queries:
    print(f"💬 User: {q}")
    response = llm_assistant.chat(q)
    print(f"🤖 AeroSync AI:\n{response}\n")
    print("-" * 70)